# ARC_ATLAS_Test_v4

Evaluate trained checkpoints on local splits and downsampled variants.

In [ ]:
from pathlib import Path
import csv
import importlib.util
import json
import time

import nibabel as nib
import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SRC = PROJECT_ROOT / 'src' / 'training_v2.py'
TEST_DIR = PROJECT_ROOT / 'data' / 'processed' / 'test_input'
LIMIT_CASES = None
USE_TTA = True
SAVE_PROBS = True
THRESHOLD = None   # None -> per-case Otsu if enabled in config; else cfg.DECISION_THRESHOLD
MIN_COMPONENT_SIZE = 0
CLOSING_ITERS = 0

# Pick run (prefer runs/latest symlink)
LATEST_LINK = PROJECT_ROOT / 'runs' / 'latest'
if LATEST_LINK.exists():
    RUN = LATEST_LINK.resolve()
else:
    RUNS = sorted((PROJECT_ROOT / 'runs').glob('20*'))
    if not RUNS:
        raise SystemExit('No runs found; train first.')
    RUN = RUNS[-1]

WEIGHTS = RUN / 'callbacks' / 'best_model_dynamic.weights.h5'
if not WEIGHTS.exists():
    alt = PROJECT_ROOT / 'runs' / 'latest_best.weights.h5'
    WEIGHTS = alt if alt.exists() else WEIGHTS
print('Using run:', RUN)
print('Weights:', WEIGHTS)

spec = importlib.util.spec_from_file_location('seg', SRC)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

cfg_json = json.load(open(RUN / 'models' / 'config.json'))
cfg_json['DATA_DIR'] = str(TEST_DIR)
cfg_json['IMAGES_DIR'] = str(TEST_DIR / 't1')
cfg_json['MASKS_DIR'] = str(TEST_DIR / 'masks')
cfg = seg.DynamicTrainingConfig(**cfg_json)
cfg.MODEL_DIR = RUN / 'models'
model = seg.build_model_for_inference(cfg, weights_path=str(WEIGHTS))

print('Model input shape:', cfg.INPUT_SHAPE)
print('Patch size:', cfg.PATCH_SIZE)


def _resolve_path(raw_value: str, manifest_path: Path) -> Path | None:
    raw = (raw_value or '').strip()
    if not raw:
        return None
    p = Path(raw)
    if p.is_absolute():
        return p if p.exists() else None
    candidates = [p, TEST_DIR / p]
    for base in manifest_path.parents:
        candidates.append(base / p)
    seen = set()
    for c in candidates:
        key = str(c)
        if key in seen:
            continue
        seen.add(key)
        if c.exists():
            return c
    return None


cases = []
manifest = TEST_DIR / 'manifest.csv'
if manifest.exists():
    with open(manifest, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t1 = _resolve_path(row.get('t1', ''), manifest)
            if t1 is None:
                continue
            msk = _resolve_path(row.get('mask', ''), manifest)
            key = row.get('key') or t1.stem
            cases.append({'key': key, 't1': t1, 'mask': msk if (msk and msk.exists()) else None})

if not cases:
    t1_dir = TEST_DIR / 't1'
    msk_dir = TEST_DIR / 'masks'
    for t1 in sorted(t1_dir.glob('*.nii.gz')):
        key = t1.name.replace('_T1w_MNI_norm.nii.gz', '').replace('.nii.gz', '')
        msk = None
        if msk_dir.exists():
            cand = msk_dir / t1.name.replace('_T1w_MNI_norm', '_lesion_mask_MNI_clean')
            if cand.exists():
                msk = cand
        cases.append({'key': key, 't1': t1, 'mask': msk})

if not cases:
    raise SystemExit(f'No test images found under {TEST_DIR}')

if LIMIT_CASES is not None:
    cases = cases[: int(LIMIT_CASES)]

pred_root = RUN / 'test_predictions'
pred_dir = pred_root / time.strftime('%Y%m%d_%H%M%S')
pred_dir.mkdir(parents=True, exist_ok=True)

rows = []
patch_size = tuple(cfg.PATCH_SIZE or cfg.INPUT_SHAPE[:-1])
for i, case in enumerate(cases, 1):
    img_obj = nib.as_closest_canonical(nib.load(str(case['t1'])))
    x = img_obj.get_fdata().astype(np.float32)

    probs = seg.gaussian_tta_predict(
        model,
        x,
        patch_size=patch_size,
        overlap=cfg.GAUSSIAN_TILE_OVERLAP,
        sigma=cfg.GAUSSIAN_TILE_SIGMA,
        tta=USE_TTA,
    )

    brain_mask = seg.compute_brain_mask(x)
    thr = THRESHOLD
    if thr is None and not bool(getattr(cfg, 'USE_PER_CASE_OTSU', True)):
        thr = float(getattr(cfg, 'DECISION_THRESHOLD', 0.1))

    pred = seg.apply_postprocessing(
        probs,
        threshold=thr,
        min_size=int(MIN_COMPONENT_SIZE),
        closing=int(CLOSING_ITERS),
        brain_mask=brain_mask,
        clamp=getattr(cfg, 'OTSU_CLAMP', (0.05, 0.25)),
        min_prob=float(getattr(cfg, 'OTSU_MIN_PROB', 0.01)),
    )
    pred_u8 = (pred > 0.5).astype(np.uint8)

    safe_key = str(case['key']).replace('/', '_').replace(' ', '_')
    pred_path = pred_dir / f"{safe_key}_pred_mask.nii.gz"
    nib.save(nib.Nifti1Image(pred_u8, img_obj.affine, img_obj.header), str(pred_path))

    prob_path = ''
    if SAVE_PROBS:
        prob_path = pred_dir / f"{safe_key}_prob.nii.gz"
        nib.save(nib.Nifti1Image(probs.astype(np.float32), img_obj.affine, img_obj.header), str(prob_path))

    row = {
        'key': case['key'],
        't1_path': str(case['t1']),
        'gt_mask_path': str(case['mask']) if case['mask'] else '',
        'pred_mask_path': str(pred_path),
        'prob_path': str(prob_path) if prob_path else '',
    }

    if case['mask'] and Path(case['mask']).exists():
        gt = nib.as_closest_canonical(nib.load(str(case['mask']))).get_fdata()
        y_true = (gt > 0.5).astype(np.float32)
        if y_true.shape != pred.shape:
            y_true = seg._center_crop_or_pad_volume(y_true, pred.shape)
        row['dice'] = float(seg.dice_soft_np(y_true, pred.astype(np.float32)))
    else:
        row['dice'] = ''

    rows.append(row)
    print(f"[{i}/{len(cases)}] {case['key']} -> {pred_path.name}" + (f" | dice={row['dice']:.4f}" if row['dice'] != '' else ''))

metrics_csv = pred_dir / 'predictions.csv'
with open(metrics_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

with_gt = [r for r in rows if r['dice'] != '']
if with_gt:
    mean_dice = float(np.mean([float(r['dice']) for r in with_gt]))
    print(f"Cases with GT masks: {len(with_gt)}/{len(rows)} | mean Dice: {mean_dice:.4f}")
else:
    print(f"No GT masks found. Generated predictions for {len(rows)} image(s).")
print('Prediction outputs:', pred_dir)
print('Summary CSV:', metrics_csv)


## (Optional) Evaluate on downsampled sets
Use helper functions from `src/downsampling/*.py` to generate degraded test sets in `data/downsampled/`, then load them with the training loader for metrics.